# Data Description

- Background: 2026-11-08學生指示游智彬資格爭議下先研究暫納入；保存官方程序原文、G6範圍查核、學生決定與來源索引。
- date received: 2026-11-08（本次取得HTTP正文與對話決定）
- Path to data file: 官方法院／全國法規資料庫、鏡週刊轉載、觀察雜誌來源URL見`source_manifest.json`；使用者決定來自本次對話。
- Unit of observation: 四份主體全文擷取文字、三份JSON（使用者納入決定、G6搜尋判定、四來源索引）。macOS的`._`旁檔不作資料或來源。
- Sample period: 來源報導2023–2026；官方程序截至2026-11-06，本次於2026-11-08核對。
- Known issues: 各txt是HTTP回應主體擷取文字，不冒稱原HTML bytes；外部原HTML存任務暫存區。鏡週刊為三立報導轉載，親屬關係非戶籍認證；觀察雜誌立場鮮明，僅採明載職涯、不採政治評論。G6=0為搜尋／範圍規則下的結果，已知四親等親屬另列。研究納入與官方資格狀態分開。
- Definition for each variable: txt為主體全文；user_inclusion_decision含候選人鍵、原始指令、標準化研究決定、官方狀態、影響假設及追蹤觸發；G6_search_and_coding含10個實際查詢、全文來源、value／label、排除線索、親等、證據層級與限制；source_manifest每列path／source_url／title／access_date／capture_method／full_main_body_read／selector／note／sha256。
- Data path: 下表7個精確路徑。
- Role: raw / source captures and explicit research decision。
- Verification status: 本Notebook只讀逐檔SHA、JSON結構、日期／候選人鍵與全文錨點；clean kernel執行保留輸出。
- SHA-256: 下表為實際保存檔案指紋。
- Next processing notebook: [暫納入處理](../code/include_you_zhibin_provisional_20261108.ipynb)。
- Author: RUEI-HSIANG, CHANG／張睿庠（B12303030；AI協助）。

| Exact raw path | SHA-256 |
| --- | --- |
| `data/raw/you_zhibin_provisional_20261108/court_20261106_fulltext.txt` | `7c0377ed5e67966b3c2c23237c9822f27325eb84641a483ac298a46fcec640f9` |
| `data/raw/you_zhibin_provisional_20261108/family_mirror_fulltext.txt` | `c4febfeb8180ab084bf5d004b0faf0f2fbce3da15e0188353a6805ba173260a2` |
| `data/raw/you_zhibin_provisional_20261108/family_observer_fulltext.txt` | `0693acac19cfc935760450d55a27baa0956f7752e1f9d7cfe36da4b9b8a14e8e` |
| `data/raw/you_zhibin_provisional_20261108/kinship_law_fulltext.txt` | `76cba03314e23af641b31dec3f1e9380dcd2e8270a9da702231cd821bb46a3de` |
| `data/raw/you_zhibin_provisional_20261108/user_inclusion_decision.json` | `2c200f01586e9e9c9cc885841dca8aea62198242710609a6cdd3526503f6084f` |
| `data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json` | `baf49b670061b8cd8454718c452bd658be7f1da60b1b29b896351589a54302ea` |
| `data/raw/you_zhibin_provisional_20261108/source_manifest.json` | `410622864ebc202fbaf1044388a66102f4d74378e824369efd0d135d6859b75a` |

In [1]:
from pathlib import Path
import json,hashlib
import pandas as pd
from IPython.display import display,Markdown
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'templates').is_dir()),None)
assert ROOT is not None

## Change for Your Own Data

僅讀取下列固定版本；不重新下載、不更動文字或判定。

In [2]:
RAW_PINS = {'data/raw/you_zhibin_provisional_20261108/court_20261106_fulltext.txt': '7c0377ed5e67966b3c2c23237c9822f27325eb84641a483ac298a46fcec640f9', 'data/raw/you_zhibin_provisional_20261108/family_mirror_fulltext.txt': 'c4febfeb8180ab084bf5d004b0faf0f2fbce3da15e0188353a6805ba173260a2', 'data/raw/you_zhibin_provisional_20261108/family_observer_fulltext.txt': '0693acac19cfc935760450d55a27baa0956f7752e1f9d7cfe36da4b9b8a14e8e', 'data/raw/you_zhibin_provisional_20261108/kinship_law_fulltext.txt': '76cba03314e23af641b31dec3f1e9380dcd2e8270a9da702231cd821bb46a3de', 'data/raw/you_zhibin_provisional_20261108/user_inclusion_decision.json': '2c200f01586e9e9c9cc885841dca8aea62198242710609a6cdd3526503f6084f', 'data/raw/you_zhibin_provisional_20261108/G6_search_and_coding.json': 'baf49b670061b8cd8454718c452bd658be7f1da60b1b29b896351589a54302ea', 'data/raw/you_zhibin_provisional_20261108/source_manifest.json': '410622864ebc202fbaf1044388a66102f4d74378e824369efd0d135d6859b75a'}
texts={};records={}
for rel,expected in RAW_PINS.items():
    p=ROOT/rel
    assert p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest()==expected,rel
    if p.suffix=='.json':records[p.name]=json.loads(p.read_text())
    else:texts[p.name]=p.read_text()
print('7份原始記錄SHA通過；4份主體文字與3份JSON。')

7份原始記錄SHA通過；4份主體文字與3份JSON。


## Summary

### Sample

In [3]:
display(pd.DataFrame(records['source_manifest.json']))
d=records['user_inclusion_decision.json']
display(pd.DataFrame([{k:d[k] for k in ['candidate_key','candidate_name','normalized_decision','official_status_preserved','research_label']}]))
for name,text in texts.items():
    display(Markdown(f'**{name}：{len(text)}字元，前200字預覽；採用前已讀完整主體。**'))
    print(text[:200])

,path,source_url,title,access_date,capture_method,full_main_body_read,selector,note,sha256
0,data/raw/you_zhibin_provisional_20261108/court...,https://www.judicial.gov.tw/tw/cp-1888-1645308...,最高行政法院115年度抗字第273號新聞稿,2026-11-08,HTTP200 HTML main-body extraction; source HTML...,True,.cp,官方程序狀態；申請登記、10/16資格不符、11/6暫時救濟抗告駁回；非本案終局判決。,7c0377ed5e67966b3c2c23237c9822f27325eb84641a48...
1,data/raw/you_zhibin_provisional_20261108/famil...,https://www.mirrormedia.mg/external/setn_1627531,三立報導之鏡週刊轉載,2026-11-08,HTTP200 HTML main-body extraction; source HTML...,True,article,全文明載外叔公游日正曾任鄉長／立委；四親等，超出已確認三親等範圍。不是戶籍認證。,c4febfeb8180ab084bf5d004b0faf0f2fbce3da15e0188...
2,data/raw/you_zhibin_provisional_20261108/famil...,https://www.observer-taipei.com/book2021/item/...,觀察雜誌2023年12月人物報導,2026-11-08,HTTP200 HTML main-body extraction; source HTML...,True,.itemFullText,立場鮮明次級人物報導；全文將外公職涯描述為軍人轉業公務員、鹽場辦公室副主任，無可採政治任命證據。,0693acac19cfc935760450d55a27baa0956f7752e1f9d7...
3,data/raw/you_zhibin_provisional_20261108/kinsh...,https://law.moj.gov.tw/LawClass/LawSingle.aspx...,民法第968條親等計算,2026-11-08,HTTP200 HTML main-body extraction; source HTML...,True,.law-article,沿現行G6親等計算；候選人→母親→外祖父→共同上一代→外叔公，四親等。,76cba03314e23af641b31dec3f1e9380dcd2e8270a9da7...


,candidate_key,candidate_name,normalized_decision,official_status_preserved,research_label
0,2026|直轄市議員|臺北市第6選舉區|游智彬,游智彬,暫將游智彬視為研究可納入的候選人，加入兩市合併與議員池；資格爭議待後續追蹤，報告揭露。,10/16中選會審定資格不符；11/6最高行政法院駁回暫時救濟抗告，本案法律爭議仍待訴訟釐清。,資格爭議／待後續確認；研究暫納入


**court_20261106_fulltext.txt：2173字元，前200字預覽；採用前已讀完整主體。**

新聞摘要：
定暫時狀態之處分，係使聲請人在本案判決確定前，可依該裁定所定暫時狀態實現其權利，而在一定範圍內達到與本案勝訴判決確定相同之結果。因此，聲請人應釋明使法院形成本案訴訟勝訴之可能性較高，才能認為有准許之必要性。因抗告人並未能釋明其本案勝訴可能性較高，經綜合衡量比較本件如暫時准許抗告人定暫時狀態處分之聲請對影響選舉結果公益所可能造成之重大嚴重損害（即選舉秩序遭受破壞、選務資源鉅額浪費及選民權


**family_mirror_fulltext.txt：1427字元，前200字預覽；採用前已讀完整主體。**

首頁
／
時事
游智彬爆靠外公寄生台灣 486：真的假的？
2025/03/22 04:55
最後更新
2026/04/02 12:26
加入為Google 偏好來源
文｜
三立新聞網
記者簡浩正／綜合報導
▲游智彬（藍衣者）被踢爆「全家28個中國人，靠外公1人寄生台灣」。（圖／翻攝自游智彬臉書）
大鬧立院的游智彬是「中國人」？國民黨推動反戒嚴與反廢死兩項公投，昨日提案列入院會議程，上午開會時，新


**family_observer_fulltext.txt：1793字元，前200字預覽；採用前已讀完整主體。**

2022
年11
月九合一選舉，游智彬打擊民進黨貪腐和論文抄襲的力道，贏得全台高度關注，無奈新黨的基礎薄弱，難敵地方派系的夾殺，加上泛藍三大戰將在桃園同一選區分散票源，游智彬以4
千多票飲恨落選。2024
立委選舉，游智彬捲土重來，勢在必得。
游智彬對民進黨連發三砲
在桃園市長和議長的鼓勵下，游智彬沒有時間失志，服務處和服務團隊依然鬥志昂揚，有別於落選熄燈的中壢舊政治模式，反而得到更多中壢鄉親的支


**kinship_law_fulltext.txt：80字元，前200字預覽；採用前已讀完整主體。**

血親親等之計算，直系血親，從己身上下數，以一世為一親等；旁系血親，從己身數至同源之直系血親，再由同源之直系血親，數至與之計算親等之血親，以其總世數為親等之數。



### Summary Stats

In [4]:
search=records['G6_search_and_coding.json']
print('來源主體數',len(texts),'JSON數',len(records),'記錄搜尋式',len(search['queries']),'範圍排除線索',len(search['excluded_leads']))
display(pd.DataFrame(search['excluded_leads']))
assert d['candidate_key']==search['candidate_key'] and d['not_an_official_approval']
assert '抗告駁回' in texts['court_20261106_fulltext.txt']
assert '外叔公游日正' in texts['family_mirror_fulltext.txt']
assert '公務員' in texts['family_observer_fulltext.txt']
assert '旁系血親' in texts['kinship_law_fulltext.txt']
for source in records['source_manifest.json']:
    assert source['full_main_body_read'] and RAW_PINS[source['path']]==source['sha256']
print('PASS：研究暫納入決定、官方程序與G6搜尋紀錄相容；不是官方合格認證。')

來源主體數 4 JSON數 3 記錄搜尋式 10 範圍排除線索 2


,relative,public_office,kinship_degree,reason,occupation
0,外叔公游日正,報導明載曾任龍潭鄉長與立委,4.0,四親等超出本研究三親等範圍；關係採次級報導，非戶籍認證。,NaN
1,外公游亞慶,NaN,NaN,一般軍職／常任文官按先前學生規則排除；未找到其擔任符合範圍之政治任命公職的可採記載。,報導記載軍人轉業公務員，漳浦鹽場辦公室副主任離休


PASS：研究暫納入決定、官方程序與G6搜尋紀錄相容；不是官方合格認證。
